### Functional / Taxonomic Analysis using mixed models

In [ ]:
import pandas as pd
import numpy as np
import re
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

In [ ]:
### define setting to indicate raw data in saved files
database = "MGnify" #MGnify / IGC / Qin
grouping = "Groups" #SubgGroups / Subgroups
new_discovery_study = "Werner" # ValdesMas / Werner
file_prefix = database +"_"+grouping + "_" + new_discovery_study + "_"
print(file_prefix)

prophane_results_path = database + "_" + grouping + "(DiscoveryJavaOutput)/prophane_results/"

sc_threshold = 5

In [ ]:
### load shared (and corrected?) metaproteins and annotation data for these metaproteins
protein_abundance = pd.read_csv(file_prefix+"shared_metaproteins(discovery_studies_normalized_MMUPHin_corrected).csv", index_col = "Unnamed: 0")
print(len(protein_abundance.index))
protein_abundance.head()

In [ ]:
# load annotation
prophane_summary = pd.read_csv(prophane_results_path+"MGnify_Groups_Werner_summary.csv", index_col = "#pg", sep=",")  ### only shared in discovery: SharedProteins(IBDDiscoveryValidationBatchCorrection).csv
protein_annotation = prophane_summary.loc[protein_abundance.index].iloc[:,0:23]
#protein_annotation.columns

In [ ]:
### load sample metadata nad matche metadata and measurement IDs
meta_data_path = r"SupplementaryFile1(WernerDiscovery)-Revision.xlsx"
sample_annotation = pd.read_excel(meta_data_path, sheet_name="SampleMetadata", index_col = "SampleID" )

sample_annotation.columns = [re.sub(r'[^A-Za-z0-9]+', '_', col) for col in sample_annotation.columns]
sample_annotation=sample_annotation.add_suffix("_", axis="columns")
protein_abundance=protein_abundance.add_suffix("_", axis="columns")

i = 0
for measurementID in protein_abundance.columns:
    #print(type(measurementID))
    for sampleID in sample_annotation.columns:
        if sampleID in measurementID:
            protein_abundance.columns.values[i] = sampleID 
    i = i + 1


In [ ]:
# function to transform the p-values in asterics
def get_significance_asterisks(p_value):
    if p_value < 0.001:
        return 'p < 0.001'
    elif p_value < 0.01:
        return 'p < 0.01'
    elif p_value < 0.05:
        return 'p < 0.05'
    else:
        return f"p = {p_value.round(2)}"  # not significant

In [ ]:
### Fig2 A-L
### Select all IBD and Control samples from discovery datasets
mask = (sample_annotation.loc["condition"] == "diseased") & (sample_annotation.loc["study"] == "Henry")
henry_diseased =sample_annotation.columns[mask].tolist()
#henry_diseased.remove("20190114_48_S01F_env_")
mask = (sample_annotation.loc["condition"] == "control") & (sample_annotation.loc["study"] == "Henry")
henry_control =sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["condition"] == "diseased") & (sample_annotation.loc["study"] == "Lehmann")
lehmann_diseased = sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["condition"] == "control") & (sample_annotation.loc["study"] == "Lehmann")
lehmann_control = sample_annotation.columns[mask].tolist()

mask = (sample_annotation.loc["condition"] == "diseased") & (sample_annotation.loc["study"] == "Thuy-Boun")
thuyboun_diseased = sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["condition"] == "control") & (sample_annotation.loc["study"] == "Thuy-Boun")
thuyboun_control = sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["condition"] == "diseased") & (sample_annotation.loc["study"] == "Lloyd-Price") & (sample_annotation.loc["UseCase"] == "BiomarkerDiscovery")
lloydprice_diseased = sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["condition"] == "control") & (sample_annotation.loc["study"] == "Lloyd-Price") & (sample_annotation.loc["UseCase"] == "BiomarkerDiscovery")
lloydprice_control = sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["condition"] == "diseased") & (sample_annotation.loc["study"] == "Werner") & (sample_annotation.loc["UseCase"] == "BiomarkerDiscovery")
werner_diseased = sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["condition"] == "control") & (sample_annotation.loc["study"] == "Werner") & (sample_annotation.loc["UseCase"] == "BiomarkerDiscovery")
werner_control = sample_annotation.columns[mask].tolist()


group_dict = {"Lloydprice (diseased)":lloydprice_diseased, "Lloydprice (control)":lloydprice_control, 
              "Thuyboun (diseased)":thuyboun_diseased, "Thuyboun (control)":thuyboun_control, 
              "Henry (diseased)":henry_diseased, "Henry (control)":henry_control, 
              "Lehmann (diseased)":lehmann_diseased,"Lehmann (control)":lehmann_control,
              "Werner (diseased)":werner_diseased,"Werner (control)":werner_control
              }

In [ ]:
### Fig 4F
### Select all CD, UC and Control samples from discovery datasets
mask = (sample_annotation.loc["disease"] == "normal") & (sample_annotation.loc["study"] == "Henry")
henry_normal =sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["disease"] == "UC") & (sample_annotation.loc["study"] == "Henry")
henry_UC =sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["disease"] == "CD") & (sample_annotation.loc["study"] == "Henry")
henry_CD =sample_annotation.columns[mask].tolist()

mask = (sample_annotation.loc["disease"] == "normal") & (sample_annotation.loc["study"] == "Lehmann")
lehmann_normal =sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["disease"] == "UC") & (sample_annotation.loc["study"] == "Lehmann") ### UCa / UCr / UC
lehmann_UC =sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["disease"] == "CD") & (sample_annotation.loc["study"] == "Lehmann")
lehmann_CD =sample_annotation.columns[mask].tolist()

mask = (sample_annotation.loc["disease"] == "normal") & (sample_annotation.loc["study"] == "Thuy-Boun")
thuyboun_normal =sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["disease"] == "UC") & (sample_annotation.loc["study"] == "Thuy-Boun")
thuyboun_UC =sample_annotation.columns[mask].tolist()

mask = (sample_annotation.loc["disease"] == "normal") & (sample_annotation.loc["study"] == "Lloyd-Price") & (sample_annotation.loc["UseCase"] == "BiomarkerDiscovery")
lloydprice_normal = sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["disease"] == "UC") & (sample_annotation.loc["study"] == "Lloyd-Price") & (sample_annotation.loc["UseCase"] == "BiomarkerDiscovery")
lloydprice_UC = sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["disease"] == "CD") & (sample_annotation.loc["study"] == "Lloyd-Price") & (sample_annotation.loc["UseCase"] == "BiomarkerDiscovery")
lloydprice_CD = sample_annotation.columns[mask].tolist()

mask = (sample_annotation.loc["disease"] == "normal") & (sample_annotation.loc["study"] == "Werner")
werner_normal =sample_annotation.columns[mask].tolist()
mask = (sample_annotation.loc["disease"] == "UC") & (sample_annotation.loc["study"] == "Werner")
werner_UC =sample_annotation.columns[mask].tolist()


group_dict = {"Lloydprice (normal)":lloydprice_normal,"Lloydprice (UC)":lloydprice_UC,"Lloydprice (CD)":lloydprice_CD,
              "Thuyboun (normal)":thuyboun_normal, "Thuyboun (UC)":thuyboun_UC, 
              "Henry (normal)":henry_normal, "Henry (UC)":henry_UC,"Henry (CD)":henry_CD, 
              "Lehmann (normal)":lehmann_normal,"Lehmann (UC)":lehmann_UC, "Lehmann (CD)":lehmann_CD,
              "Werner (normal)":werner_normal, "Werner (UC)":werner_UC,
              }

In [ ]:
### optional: filter only for bacterial proteins (Figure 2 E-H)
# get only microbial proteins
mask = (protein_annotation.T.loc["task_1_Taxonomic_Annotation_Task_1_superkingdom"] == "Bacteria")
microbial_protein_abundance = protein_abundance.loc[mask]


In [ ]:
# Fig 2 (individual figures)
#
#annotation_list = ["Homo sapiens"]
#annotation_type = "task_1_Taxonomic_Annotation_Task_1_species"

# Fig 2 B
#annotation_list = ["Faecalibacterium prausnitzii"]
#annotation_type = "task_1_Taxonomic_Annotation_Task_1_species"

# Fig 2 C
#annotation_list = ["Bacteroidetes"]
#annotation_type = "task_1_Taxonomic_Annotation_Task_1_phylum"

# Fig 2 D
#annotation_list = ["Enterobacteriaceae"]
#annotation_type = "task_1_Taxonomic_Annotation_Task_1_family"


# Fig 2 E
#annotation_list = ['R00238',"R03027","R01778","R01178","R01365","R01174","R01688","R01179","R05576"] # butyrate_fermentation
#annotation_type = "Kegg_Reaction"
# Fig 2 F
#annotation_list = ["R00315","R00230"] # acetate_fermentation
#annotation_type = "Kegg_Reaction"
# Fig 2 G
#annotation_list = ["R00930","R00833","R0344","R00342","R01082","R02164","R00928","R00432","R02765","R01859","R11674","R00214","R00217"] # propionate_fermentation
#annotation_type = "Kegg_Reaction"
# Fig 2 H
#annotation_list = ["R02433","R07136","R00782","R07634","R00897"] # cystein_metabolism
#annotation_type = "Kegg_Reaction"
# Fig 2 I
#annotation_list = ["GO:1990266"] # neutrophil migration
#annotation_type = "GO_Term"
# Fig 2 J
#annotation_list = ["immunoglobulin"]
#annotation_type = "task_0_Functional_Annotation_Task_1_desc"
# Fig 2 K
#annotation_list = ["GO:0004252"] # serine-type endopeptidase activity
#annotation_type = "GO_Term"
# Fig 2 L
#annotation_list = ["GO:0030036"]
#annotation_type = "GO_Term"

#Fig 4F
annotation_list =  ["Annexin A2"] 
annotation_type = "task_0_Functional_Annotation_Task_1_desc"


# Supplementary Figures
#annotation_list = ["Blautia"]
#annotation_type = "task_1_Taxonomic_Annotation_Task_1_genus"

# Supplementary Figure X
#annotation_list = ["Ruminococcus"]
#annotation_type = "task_1_Taxonomic_Annotation_Task_1_genus"

# Supplementary Figure X
#annotation_list = ["Bifidobacterium"]
#annotation_type = "task_1_Taxonomic_Annotation_Task_1_genus"

# Supplementary Figure X
#annotation_list = ["Firmicutes"]
#annotation_type = "task_1_Taxonomic_Annotation_Task_1_phylum"

In [ ]:
# extract the summed abundances of all metaproteins with a given annotation 
def extract_reaction_abundance_revised(samplelist, dataframe,annotation_type, annotation_list):
    protein_abundance=dataframe.loc[:,samplelist]
    #maske = protein_annotation[annotation_type].str.contains(annotation_id, case=False, na=False)
    pattern = r"\b(?:" + "|".join(map(re.escape, annotation_list)) + r")\b"

    maske = protein_annotation[annotation_type].str.contains(pattern, case=False, na=False, regex=True)
    summen = protein_abundance[maske].sum()

    return summen

In [ ]:
# for each group in list of patient-batch groups: extract abundance
# create boxplot
pathway_group_abundances = []
group_labels = []

# for each patient group collect and add abundances for each ID, than append to the final list
for key in group_dict: # individual studies: group_dict1-4
    group_labels.append(key)
    # set initial zero value array
    group_abundances = np.zeros(len(group_dict[key]))
    #for id in annotation_list:
    #    #group_abundances.append()
    group_abundances=np.add(group_abundances, extract_reaction_abundance_revised(group_dict[key],protein_abundance,annotation_type,annotation_list))
    pathway_group_abundances.append(group_abundances)


In [ ]:
### create input for linear mixed effect modeling
df_abundance = pd.DataFrame(pd.concat(pathway_group_abundances, axis=0).T)*100
df_abundance['sample'] = df_abundance.index

# Fig2 A-L (for IBD vs healthy analysis)
l1 = ["study","condition"]

# Fig 4F (for CD- and UC-specific analysis)
#l1 = ["study","disease"]

disease_study_df = sample_annotation.loc[l1].T
modeling_df = pd.merge(df_abundance, disease_study_df, how='left', left_on='sample', right_on=disease_study_df.index)
modeling_df = modeling_df.rename(columns={modeling_df.columns[0]: "abundance"})
modeling_df

In [ ]:
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import ConvergenceWarning
### get p-value for each annotation
### linear mixed models with and without healthy/diseased and study as random slope random intercept


# Fig 2A-L
md1=smf.mixedlm("abundance ~ 1 + condition", data = modeling_df, groups=modeling_df["study"], re_formula= "~1 + condition") 

# Fig 4F
#md1=smf.mixedlm("abundance ~ 1 + disease", data = modeling_df, groups=modeling_df["study"], re_formula= "~1 + disease") 

mdf = md1.fit(method=["lbfgs"], reml=False)
print(mdf.summary())
#md1=smf.mixedlm("Abundance ~ 1", data = dataframe, groups=dataframe["Study"], re_formula = "~1 + disease")
#print(mdf.summary())


In [ ]:
# Fig 2A-L
mdf.pvalues["condition[T.diseased]"]
# Fig 4F
#mdf.pvalues["disease[T.normal]"]



In [ ]:
### individual figure
gruppen1 = np.repeat([1, 2, 3, 4, 5], [len(pathway_group_abundances[0]), len(pathway_group_abundances[2]), len(pathway_group_abundances[4]), len(pathway_group_abundances[6]), len(pathway_group_abundances[8])])
gruppen2 = np.repeat([1, 2, 3, 4, 5], [len(pathway_group_abundances[1]), len(pathway_group_abundances[3]), len(pathway_group_abundances[5]), len(pathway_group_abundances[7]), len(pathway_group_abundances[9])])

diseased = pd.concat([pathway_group_abundances[0], pathway_group_abundances[2], pathway_group_abundances[4], pathway_group_abundances[6],pathway_group_abundances[8] ])*100
healthy = pd.concat([pathway_group_abundances[1], pathway_group_abundances[3], pathway_group_abundances[5], pathway_group_abundances[7],pathway_group_abundances[9] ])*100

# Farben
farben = {
    1: "#cab2d6",
    2: "#33a02c",
    3: "#ff0000",
    4: "#4472c4",
    5: "#fcca27"
}

# Position der vier Gruppen relativ zum Boxplot
gruppen_position1 = {
    1: 0.80,
    2: 0.90,
    3: 1.00,
    4: 1.10,
    5: 1.20
}
gruppen_position2 = {
    1: 1.80,
    2: 1.90,
    3: 2.00,
    4: 2.10,
    5: 2.20
}

fig, ax = plt.subplots(figsize=(6, 8))

# Boxplot
ax.boxplot(
    [diseased, healthy],
    positions=[1,2],
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", alpha=0.4),
    medianprops=dict(color="black", linewidth=2),
    showfliers= False
    
)

# Punkte zeichnen
for gruppe in np.unique(gruppen1):
    mask = gruppen1 == gruppe

    # Nur kleiner Jitter innerhalb der Gruppe
    x = (
        np.full(np.sum(mask), gruppen_position1[gruppe])
        + np.random.uniform(-0.01, 0.01, np.sum(mask))
    )

    ax.scatter(
        x,
        diseased[mask],
        color=farben[gruppe],
        s=60,
        alpha=0.8,
        label=f"Gruppe {gruppe}"
    )
    
for gruppe in np.unique(gruppen2):
    mask = gruppen2 == gruppe

    # Nur kleiner Jitter innerhalb der Gruppe
    x = (
        np.full(np.sum(mask), gruppen_position2[gruppe])
        + np.random.uniform(-0.01, 0.01, np.sum(mask))
    )

    ax.scatter(
        x,
        healthy[mask],
        color=farben[gruppe],
        s=60,
        alpha=0.8,
        label=f"Gruppe {gruppe}"
    )

#sig = get_significance_asterisks(mdf.pvalues["condition[T.diseased]"])
#ax.text(
#    1.35, max(modeling_df.abundance),                     # (x, y)-Position im Plot
#    f"{sig}",         # Inhalt der Box
#    fontsize=15,
#    color='black',
#    bbox=dict(
#        facecolor='white',  # Hintergrundfarbe
#        edgecolor='lightgrey',       # Rahmenfarbe
#        boxstyle='round,pad=0.2', # Box-Stil
#        linewidth=2
#    ))

### y axis
# set y axis label & limit 
ax.set_ylabel('Relative Abundance in [%]', fontsize=15)
ax.set_ylim(top= max(modeling_df.abundance)*1.1)
plt.yticks(fontsize=12)

ax.set_xticks([1,2])
ax.set_xticklabels(["IBD", "Healthy"], fontsize = 20)

#ax.legend(title="Gruppen")
#plt.title(annotation_list[0])
if "Taxonomic" in annotation_type:
    plt.title(annotation_list[0], style='italic') # '$\it{text you want to show in italics}$'
else:
    plt.title(annotation_list[0])
plt.savefig(f"{re.sub(r'[:]', '', annotation_list[0])}.png", bbox_inches="tight")
plt.show()


In [ ]:
### Fig 4F
gruppen1 = np.repeat([1, 2, 3, 4, 5], [len(pathway_group_abundances[0]), len(pathway_group_abundances[3]), len(pathway_group_abundances[5]), len(pathway_group_abundances[8]), len(pathway_group_abundances[11])])
gruppen2 = np.repeat([1, 2, 3, 4, 5], [len(pathway_group_abundances[1]), len(pathway_group_abundances[4]), len(pathway_group_abundances[6]), len(pathway_group_abundances[9]), len(pathway_group_abundances[12])])
gruppen3 = np.repeat([1, 2, 3], [len(pathway_group_abundances[2]), len(pathway_group_abundances[7]), len(pathway_group_abundances[10])])

healthy = pd.concat([pathway_group_abundances[0], pathway_group_abundances[3], pathway_group_abundances[5], pathway_group_abundances[8],pathway_group_abundances[11] ])*100
UC = pd.concat([pathway_group_abundances[1], pathway_group_abundances[4], pathway_group_abundances[6], pathway_group_abundances[9],pathway_group_abundances[12] ])*100
CD = pd.concat([pathway_group_abundances[2], pathway_group_abundances[7], pathway_group_abundances[10] ])*100
# Farben
farben = {
    1: "#cab2d6",
    2: "#33a02c",
    3: "#ff0000",
    4: "#4472c4",
    5: "#fcca27"
}

# Position der vier Gruppen relativ zum Boxplot
gruppen_position1 = {
    1: 0.80,
    2: 0.90,
    3: 1.00,
    4: 1.10,
    5: 1.20
}
gruppen_position2 = {
    1: 1.80,
    2: 1.90,
    3: 2.00,
    4: 2.10,
    5: 2.20
}

gruppen_position3 = {
    1: 2.90,
    2: 3.00,
    3: 3.10,
}

fig, ax = plt.subplots(figsize=(6, 8))

# Boxplot
ax.boxplot(
    [healthy, UC, CD],
    positions=[1,2,3],
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", alpha=0.4),
    medianprops=dict(color="black", linewidth=2),
    showfliers= False
    
)

# Punkte zeichnen
for gruppe in np.unique(gruppen1):
    mask = gruppen1 == gruppe

    # Nur kleiner Jitter innerhalb der Gruppe
    x = (
        np.full(np.sum(mask), gruppen_position1[gruppe])
        + np.random.uniform(-0.01, 0.01, np.sum(mask))
    )

    ax.scatter(
        x,
        healthy[mask],
        color=farben[gruppe],
        s=60,
        alpha=0.8,
        label=f"Gruppe {gruppe}"
    )
    
for gruppe in np.unique(gruppen2):
    mask = gruppen2 == gruppe

    # Nur kleiner Jitter innerhalb der Gruppe
    x = (
        np.full(np.sum(mask), gruppen_position2[gruppe])
        + np.random.uniform(-0.01, 0.01, np.sum(mask))
    )

    ax.scatter(
        x,
        UC[mask],
        color=farben[gruppe],
        s=60,
        alpha=0.8,
        label=f"Gruppe {gruppe}"
    )

for gruppe in np.unique(gruppen3):
    mask = gruppen3 == gruppe

    # Nur kleiner Jitter innerhalb der Gruppe
    x = (
        np.full(np.sum(mask), gruppen_position3[gruppe])
        + np.random.uniform(-0.01, 0.01, np.sum(mask))
    )

    ax.scatter(
        x,
        CD[mask],
        color=farben[gruppe],
        s=60,
        alpha=0.8,
        label=f"Gruppe {gruppe}"
    )
### y axis
# set y axis label & limit 
ax.set_ylabel('Relative Abundance in [%]', fontsize=20)
ax.set_ylim(top= max(modeling_df.abundance)*1.1)
plt.yticks(fontsize=20)

ax.set_xticks([1,2,3])
ax.set_xticklabels(["Healthy", "UC" , "CD"], fontsize = 20)

#ax.legend(title="Gruppen")
#plt.title(annotation_list[0])
if "Taxonomic" in annotation_type:
    plt.title(annotation_list[0], style='italic', fontsize =22) # '$\it{text you want to show in italics}$'
else:
    plt.title(annotation_list[0], fontsize =22)
plt.savefig(f"{re.sub(r'[:]', '', annotation_list[0])}.png", bbox_inches="tight")
plt.show()


## Loop through all hypothesis:


In [ ]:
##### Loop thorugh all hypothesis (p-values not adjusted)
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import ConvergenceWarning

hypothesis_p_values= []
figure_list= []
hypotheses_dict = {"Homo sapiens":[["Homo sapiens"],"task_1_Taxonomic_Annotation_Task_1_species"], 
                    "Faecalibacterium prausnitzii":[["Faecalibacterium prausnitzii"], "task_1_Taxonomic_Annotation_Task_1_species"],
                    "Bacteroidetes":[["Bacteroidetes"],  "task_1_Taxonomic_Annotation_Task_1_phylum" ],
                    "Enterobacteriaceae":[["Enterobacteriaceae"], "task_1_Taxonomic_Annotation_Task_1_family"],
                    "Butyrate fermentation":[['R00238',"R03027","R01778","R01178","R01365","R01174","R01688","R01179","R05576"], "Kegg_Reaction"],
                    "Acetate fermentation":[["R00315","R00230"], "Kegg_Reaction"],
                    "Propionate fermentation":[["R00930","R00833","R0344","R00342","R01082","R02164","R00928","R00432","R02765","R01859","R11674","R00214","R00217"], "Kegg_Reaction"],
                    "Cystein metabolism":[["R02433","R07136","R00782","R07634","R00897"],"Kegg_Reaction"],
                    "Neutrophil migration":[["GO:1990266"], "GO_Term"],
                    "Immunoglobulins":[["immunoglobulin"], "task_0_Functional_Annotation_Task_1_desc"],
                    "Serine-type endopeptidase activity":[["GO:0004252"],"GO_Term"],
                    "Actin cytoskeleton organization":[["GO:0030036"],"GO_Term"],
                    "Blautia":[["Blautia"],"task_1_Taxonomic_Annotation_Task_1_genus"],
                    "Ruminococcus":[["Ruminococcus"],"task_1_Taxonomic_Annotation_Task_1_genus"],
                    "Bifidobacterium":[["Bifidobacterium"],"task_1_Taxonomic_Annotation_Task_1_genus"],
                    "Firmicutes":[["Firmicutes"],"task_1_Taxonomic_Annotation_Task_1_phylum"]}
# for each group in list of patient-batch groups: extract abundance
# create boxplot

for key, value in hypotheses_dict.items():
    annotation_type = value[1] 
    annotation_list = value[0]
    plot_title = key
    pathway_group_abundances = []
    group_labels = []

    # for each patient group collect and add abundances for each ID, than append to the final list
    for key in group_dict: # individual studies: group_dict1-4
        group_labels.append(key)
        # set initial zero value array
        group_abundances = np.zeros(len(group_dict[key]))
        #for id in annotation_list:
        #    #group_abundances.append()
        group_abundances=np.add(group_abundances, extract_reaction_abundance_revised(group_dict[key],protein_abundance,annotation_type,annotation_list))
        pathway_group_abundances.append(group_abundances)

    #pd.concat([pathway_group_abundances[0], pathway_group_abundances[2]])
    ### create input for linear mixed effect modeling
    df_abundance = pd.DataFrame(pd.concat(pathway_group_abundances, axis=0).T)*100
    df_abundance['sample'] = df_abundance.index

    # Fig2 A-L (for IBD vs healthy analysis)
    l1 = ["study","condition"]

    # Fig 4F (for CD- and UC-specific analysis)
    #l1 = ["study","disease"]

    disease_study_df = sample_annotation.loc[l1].T
    modeling_df = pd.merge(df_abundance, disease_study_df, how='left', left_on='sample', right_on=disease_study_df.index)
    modeling_df = modeling_df.rename(columns={modeling_df.columns[0]: "abundance"})
    modeling_df
    
    ### get p-value for each annotation
    ### linear mixed models with and without healthy/diseased and study as random slope random intercept


    # Fig 2A-L
    md1=smf.mixedlm("abundance ~ 1 + condition", data = modeling_df, groups=modeling_df["study"], re_formula= "~1 + condition") 

    # Fig 4F
    #md1=smf.mixedlm("abundance ~ 1 + disease", data = modeling_df, groups=modeling_df["study"], re_formula= "~1 + disease") 

    mdf = md1.fit(method=["lbfgs"], reml=False)
    if np.isnan(mdf.pvalues["condition[T.diseased]"]):
        mdf = md1.fit(method=["bfgs"], reml=False)
        if np.isnan(mdf.pvalues["condition[T.diseased]"]):
            mdf = md1.fit(method=["powell"], reml=False)

    print(mdf.summary())
    #md1=smf.mixedlm("Abundance ~ 1", data = dataframe, groups=dataframe["Study"], re_formula = "~1 + disease")
    # Fig 2A-L
    mdf.pvalues["condition[T.diseased]"]

    # Fig 4F
    #mdf.pvalues["disease[T.normal]"]

    gruppen1 = np.repeat([1, 2, 3, 4, 5], [len(pathway_group_abundances[0]), len(pathway_group_abundances[2]), len(pathway_group_abundances[4]), len(pathway_group_abundances[6]), len(pathway_group_abundances[8])])
    gruppen2 = np.repeat([1, 2, 3, 4, 5], [len(pathway_group_abundances[1]), len(pathway_group_abundances[3]), len(pathway_group_abundances[5]), len(pathway_group_abundances[7]), len(pathway_group_abundances[9])])

    diseased = pd.concat([pathway_group_abundances[0], pathway_group_abundances[2], pathway_group_abundances[4], pathway_group_abundances[6],pathway_group_abundances[8] ])*100
    healthy = pd.concat([pathway_group_abundances[1], pathway_group_abundances[3], pathway_group_abundances[5], pathway_group_abundances[7],pathway_group_abundances[9] ])*100

    # Farben
    farben = {
    1: "#cab2d6",
    2: "#33a02c",
    3: "#ff0000",
    4: "#4472c4",
    5: "#fcca27"
    }

    # Position der vier Gruppen relativ zum Boxplot
    gruppen_position1 = {
    1: 0.80,
    2: 0.90,
    3: 1.00,
    4: 1.10,
    5: 1.20
    }
    gruppen_position2 = {
    1: 1.80,
    2: 1.90,
    3: 2.00,
    4: 2.10,
    5: 2.20
    }

    fig, ax = plt.subplots(figsize=(6, 8))

    # Boxplot
    ax.boxplot(
    [diseased, healthy],
    positions=[1,2],
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", alpha=0.4),
    medianprops=dict(color="black", linewidth=2),
    showfliers= False
    
    )

    # Punkte zeichnen
    for gruppe in np.unique(gruppen1):
        mask = gruppen1 == gruppe

        # Nur kleiner Jitter innerhalb der Gruppe
        x = (
            np.full(np.sum(mask), gruppen_position1[gruppe])
            + np.random.uniform(-0.01, 0.01, np.sum(mask))
        )

        ax.scatter(
            x,
            diseased[mask],
            color=farben[gruppe],
            s=60,
            alpha=0.8,
            label=f"Gruppe {gruppe}"
        )
    
    for gruppe in np.unique(gruppen2):
        mask = gruppen2 == gruppe

        # Nur kleiner Jitter innerhalb der Gruppe
        x = (
            np.full(np.sum(mask), gruppen_position2[gruppe])
            + np.random.uniform(-0.01, 0.01, np.sum(mask))
        )

        ax.scatter(
            x,
            healthy[mask],
            color=farben[gruppe],
            s=60,
            alpha=0.8,
            label=f"Gruppe {gruppe}"
        )

    sig = get_significance_asterisks(mdf.pvalues["condition[T.diseased]"])
    ax.text(
        1.2, max(modeling_df.abundance),                     # (x, y)-Position im Plot
        f"{sig}",         # Inhalt der Box
        fontsize=20,
        color='black',
        bbox=dict(
            facecolor='white',  # Hintergrundfarbe
            edgecolor='lightgrey',       # Rahmenfarbe
            boxstyle='round,pad=0.2', # Box-Stil
            linewidth=2
        ))

    ### y axis
    # set y axis label & limit 
    ax.set_ylabel('Relative Abundance in [%]', fontsize=20)
    ax.set_ylim(top= max(modeling_df.abundance)*1.1)
    plt.yticks(fontsize=20)
    ax.set_xticks([1,2])
    ax.set_xticklabels(["IBD", "Healthy"], fontsize = 20)

    #ax.legend(title="Gruppen")
    #plt.title(annotation_list[0])
    if "Taxonomic" in annotation_type:
        plt.title(plot_title, style='italic', fontsize=22) # italic style for taxon names
    else:
        plt.title(plot_title, fontsize=22)
    plt.savefig(f"{re.sub(r'[:]', '', annotation_list[0])}.png", bbox_inches="tight")
    figure_list.append(fig)
    hypothesis_p_values.append(mdf.pvalues["condition[T.diseased]"])
    plt.show()




In [ ]:
### same figure with corrected p-values

import statsmodels.stats.multitest as mt
hypothesis_p_values_corrected = mt.multipletests(hypothesis_p_values, method = "fdr_bh")[1]

figure_list= []
hypotheses_dict = {"Homo sapiens":[["Homo sapiens"],"task_1_Taxonomic_Annotation_Task_1_species"], 
                    "Faecalibacterium prausnitzii":[["Faecalibacterium prausnitzii"], "task_1_Taxonomic_Annotation_Task_1_species"],
                    "Bacteroidetes":[["Bacteroidetes"],  "task_1_Taxonomic_Annotation_Task_1_phylum" ],
                    "Enterobacteriaceae":[["Enterobacteriaceae"], "task_1_Taxonomic_Annotation_Task_1_family"],
                    "Butyrate fermentation":[['R00238',"R03027","R01778","R01178","R01365","R01174","R01688","R01179","R05576"], "Kegg_Reaction"],
                    "Acetate fermentation":[["R00315","R00230"], "Kegg_Reaction"],
                    "Propionate fermentation":[["R00930","R00833","R0344","R00342","R01082","R02164","R00928","R00432","R02765","R01859","R11674","R00214","R00217"], "Kegg_Reaction"],
                    "Cystein metabolism":[["R02433","R07136","R00782","R07634","R00897"],"Kegg_Reaction"],
                    "Neutrophil migration":[["GO:1990266"], "GO_Term"],
                    "Immunoglobulins":[["immunoglobulin"], "task_0_Functional_Annotation_Task_1_desc"],
                    "Serine-type endopeptidase activity":[["GO:0004252"],"GO_Term"],
                    "Actin cytoskeleton organization":[["GO:0030036"],"GO_Term"],
                    "Blautia":[["Blautia"],"task_1_Taxonomic_Annotation_Task_1_genus"],
                    "Ruminococcus":[["Ruminococcus"],"task_1_Taxonomic_Annotation_Task_1_genus"],
                    "Bifidobacterium":[["Bifidobacterium"],"task_1_Taxonomic_Annotation_Task_1_genus"],
                    "Firmicutes":[["Firmicutes"],"task_1_Taxonomic_Annotation_Task_1_phylum"]}
# for each group in list of patient-batch groups: extract abundance
# create boxplot
i = 0
for key, value in hypotheses_dict.items():
    annotation_type = value[1] 
    annotation_list = value[0]
    plot_title = key
    pathway_group_abundances = []
    group_labels = []

    # for each patient group collect and add abundances for each ID, than append to the final list
    for key in group_dict: # individual studies: group_dict1-4
        group_labels.append(key)
        # set initial zero value array
        group_abundances = np.zeros(len(group_dict[key]))
        #for id in annotation_list:
        #    #group_abundances.append()
        group_abundances=np.add(group_abundances, extract_reaction_abundance_revised(group_dict[key],protein_abundance,annotation_type,annotation_list))
        pathway_group_abundances.append(group_abundances)

    #pd.concat([pathway_group_abundances[0], pathway_group_abundances[2]])
    ### create input for linear mixed effect modeling
    df_abundance = pd.DataFrame(pd.concat(pathway_group_abundances, axis=0).T)*100
    df_abundance['sample'] = df_abundance.index

    # Fig2 A-L (for IBD vs healthy analysis)
    l1 = ["study","condition"]

    # Fig 4F (for CD- and UC-specific analysis)
    #l1 = ["study","disease"]

    disease_study_df = sample_annotation.loc[l1].T
    modeling_df = pd.merge(df_abundance, disease_study_df, how='left', left_on='sample', right_on=disease_study_df.index)
    modeling_df = modeling_df.rename(columns={modeling_df.columns[0]: "abundance"})
    modeling_df
    
    ### get p-value for each annotation
    ### linear mixed models with and without healthy/diseased and study as random slope random intercept


    # Fig 2A-L
    md1=smf.mixedlm("abundance ~ 1 + condition", data = modeling_df, groups=modeling_df["study"], re_formula= "~1 + condition") 

    # Fig 4F
    #md1=smf.mixedlm("abundance ~ 1 + disease", data = modeling_df, groups=modeling_df["study"], re_formula= "~1 + disease") 

    mdf = md1.fit(method=["lbfgs"], reml=False)
    print(mdf.summary())
    #md1=smf.mixedlm("Abundance ~ 1", data = dataframe, groups=dataframe["Study"], re_formula = "~1 + disease")
    # Fig 2A-L
    mdf.pvalues["condition[T.diseased]"]

    # Fig 4F
    #mdf.pvalues["disease[T.normal]"]

    gruppen1 = np.repeat([1, 2, 3, 4, 5], [len(pathway_group_abundances[0]), len(pathway_group_abundances[2]), len(pathway_group_abundances[4]), len(pathway_group_abundances[6]), len(pathway_group_abundances[8])])
    gruppen2 = np.repeat([1, 2, 3, 4, 5], [len(pathway_group_abundances[1]), len(pathway_group_abundances[3]), len(pathway_group_abundances[5]), len(pathway_group_abundances[7]), len(pathway_group_abundances[9])])

    diseased = pd.concat([pathway_group_abundances[0], pathway_group_abundances[2], pathway_group_abundances[4], pathway_group_abundances[6],pathway_group_abundances[8] ])*100
    healthy = pd.concat([pathway_group_abundances[1], pathway_group_abundances[3], pathway_group_abundances[5], pathway_group_abundances[7],pathway_group_abundances[9] ])*100

    # Farben
    farben = {
    1: "#cab2d6",
    2: "#33a02c",
    3: "#ff0000",
    4: "#4472c4",
    5: "#fcca27"
    }

    # Position der vier Gruppen relativ zum Boxplot
    gruppen_position1 = {
    1: 0.80,
    2: 0.90,
    3: 1.00,
    4: 1.10,
    5: 1.20
    }
    gruppen_position2 = {
    1: 1.80,
    2: 1.90,
    3: 2.00,
    4: 2.10,
    5: 2.20
    }

    fig, ax = plt.subplots(figsize=(6, 8))

    # Boxplot
    ax.boxplot(
    [diseased, healthy],
    positions=[1,2],
    widths=0.5,
    patch_artist=True,
    boxprops=dict(facecolor="lightgray", alpha=0.4),
    medianprops=dict(color="black", linewidth=2),
    showfliers= False
    
    )

    # Punkte zeichnen
    for gruppe in np.unique(gruppen1):
        mask = gruppen1 == gruppe

        # Nur kleiner Jitter innerhalb der Gruppe
        x = (
            np.full(np.sum(mask), gruppen_position1[gruppe])
            + np.random.uniform(-0.01, 0.01, np.sum(mask))
        )

        ax.scatter(
            x,
            diseased[mask],
            color=farben[gruppe],
            s=60,
            alpha=0.8,
            label=f"Gruppe {gruppe}"
        )
    
    for gruppe in np.unique(gruppen2):
        mask = gruppen2 == gruppe

        # Nur kleiner Jitter innerhalb der Gruppe
        x = (
            np.full(np.sum(mask), gruppen_position2[gruppe])
            + np.random.uniform(-0.01, 0.01, np.sum(mask))
        )

        ax.scatter(
            x,
            healthy[mask],
            color=farben[gruppe],
            s=60,
            alpha=0.8,
            label=f"Gruppe {gruppe}"
        )

    sig = get_significance_asterisks(hypothesis_p_values_corrected[i])
    ax.text(
        1.2, max(modeling_df.abundance),                     # (x, y)-Position im Plot
        f"{sig}",         # Inhalt der Box
        fontsize=20,
        color='black',
        bbox=dict(
            facecolor='white',  # Hintergrundfarbe
            edgecolor='lightgrey',       # Rahmenfarbe
            boxstyle='round,pad=0.2', # Box-Stil
            linewidth=2
        ))

    ### y axis
    # set y axis label & limit 
    ax.set_ylabel('Relative Abundance in [%]', fontsize=20)
    ax.set_ylim(top= max(modeling_df.abundance)*1.1)
    plt.yticks(fontsize=20)
    ax.set_xticks([1,2])
    ax.set_xticklabels(["IBD", "Healthy"], fontsize = 20)

    #ax.legend(title="Gruppen")
    #plt.title(annotation_list[0])
    if "Taxonomic" in annotation_type:
        plt.title(plot_title, style='italic', fontsize=22) # italic style for taxon names
    else:
        plt.title(plot_title, fontsize=22)
    plt.savefig(f"{re.sub(r'[:]', '', annotation_list[0])}.png", bbox_inches="tight")
    figure_list.append(fig)
    hypothesis_p_values.append(mdf.pvalues["condition[T.diseased]"])
    plt.show()
    i= i+1




In [ ]:
import matplotlib.pyplot as plt
import string
from io import BytesIO
from PIL import Image

import matplotlib.pyplot as plt
from matplotlib.image import imread
from io import BytesIO
import string
### creation of summary for Figure 2

# fig_list contains 12 existing matplotlib Figure objects
# fig_list = [fig1, fig2, ..., fig12]
fig_list = figure_list[0:12]


row_titles = [
    "Taxonomic composition",
    "Functional composition (microbial)",
    "Functional composition (human)",
]

summary_fig, axes = plt.subplots(
    3, 4,
    figsize=(11, 11)
)

axes = axes.flatten()

for i, (old_fig, ax) in enumerate(zip(fig_list, axes)):
    buffer = BytesIO()
    old_fig.savefig(
        buffer,
        format="png",
        dpi=300,
        bbox_inches="tight")

    buffer.seek(0)

    # Read image
    img = imread(buffer)
    ax.imshow(img)
    ax.axis("off")

    # Panel label A, B, ..., L
    ax.text(
        -0.02, 0.99,
        string.ascii_uppercase[i],
        transform=ax.transAxes,
        fontsize=18,
        fontweight="bold",
        ha="left",
        va="top",
        color="black"
    )
# insert row titles

row_title_y = [0.955, 0.635, 0.315]

for title, y in zip(row_titles, row_title_y):

    summary_fig.text(
        0.5,              # horizontal center of entire figure
        y,
        title,
        ha="center",
        va="center",
        fontsize=20,
        fontweight="bold"
    )
### adjust layout

# Legend: Sample categories
legend_elements_category = [
    Line2D([0], [0], marker='o', color='w', label='Lehmann', markerfacecolor='#4472c4', markeredgecolor='#4472c4', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Thuy-Boun', markerfacecolor='#33a02c', markeredgecolor='#33a02c', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Henry', markerfacecolor='#ff0000', markeredgecolor='#ff0000', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Lloyd-Price', markerfacecolor='#cab2d6', markeredgecolor='#cab2d6', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Werner', markerfacecolor='#fcca27', markeredgecolor='#fcca27', markersize=10)
]
texts = []

handles = legend_elements_category
labels = [legend.get_label() for legend in handles]

summary_fig.legend(handles=handles, 
           labels=labels, 
           #title='Study', 
            loc='lower center',
           ncol=5, 
           #bbox_to_anchor=(1.05, 1.0)
          )#bbox_to_anchor=(0.0, 1)

plt.subplots_adjust(
    left=0.02,
    right=0.98,
    bottom=0.03,
    top=0.93,
    wspace=0.01,
    hspace=0.2
)
plt.savefig(file_prefix + "summary.png", bbox_inches="tight", dpi=400)
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import string
from io import BytesIO
from PIL import Image
import matplotlib.pyplot as plt
from matplotlib.image import imread
from io import BytesIO
import string


### Creation of Supplementary Figure containing only 4 taxonomies:

# fig_list = [fig1, fig2, ..., fig12]
fig_list = figure_list[12:]

summary_fig, axes = plt.subplots(
    2, 2,
    figsize=(11, 11)
)

axes = axes.flatten()
for i, (old_fig, ax) in enumerate(zip(fig_list, axes)):

    # Render original figure into memory
    buffer = BytesIO()

    old_fig.savefig(
        buffer,
        format="png",
        dpi=300,
        bbox_inches="tight"
    )

    buffer.seek(0)

    # Read image
    img = imread(buffer)
    ax.imshow(img)
    ax.axis("off")

    # Panel label A, B, ..., L
    ax.text(
        -0.02, 0.99,
        string.ascii_uppercase[i],
        transform=ax.transAxes,
        fontsize=18,
        fontweight="bold",
        ha="left",
        va="top",
        color="black"
    )

# adjust layout:
# Legend: Sample categories
legend_elements_category = [
    Line2D([0], [0], marker='o', color='w', label='Lehmann', markerfacecolor='#4472c4', markeredgecolor='#4472c4', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Thuy-Boun', markerfacecolor='#33a02c', markeredgecolor='#33a02c', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Henry', markerfacecolor='#ff0000', markeredgecolor='#ff0000', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Lloyd-Price', markerfacecolor='#cab2d6', markeredgecolor='#cab2d6', markersize=10),
    Line2D([0], [0], marker='o', color='w', label='Werner', markerfacecolor='#fcca27', markeredgecolor='#fcca27', markersize=10)
]
texts = []

handles = legend_elements_category
labels = [legend.get_label() for legend in handles]

summary_fig.legend(handles=handles, 
           labels=labels, 
           #title='Study', 
            loc='upper center',
           ncol=5, 
           #bbox_to_anchor=(1.05, 1.0)
          )#bbox_to_anchor=(0.0, 1)

plt.subplots_adjust(
    left=0.02,
    right=0.98,
    bottom=0.03,
    top=0.93,
    wspace=0.01,
    hspace=0.2
)
plt.savefig(file_prefix + "SupplementaryFigure.png", bbox_inches="tight", dpi=400)
plt.show()